<a href="https://colab.research.google.com/github/draketh818199/AI-image-generator/blob/main/HuggingFace_Video_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Install the latest Hugging Face Diffusers implementation
!pip install -q -U git+https://github.com/huggingface/diffusers.git

# Install remaining dependencies
!pip install -q -U transformers accelerate sentencepiece ftfy
!pip install -q -U imageio imageio-ffmpeg


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:

!pip install --upgrade "protobuf>=7.36.1" "sentencepiece>=0.2.0"

  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [3]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU."
    )


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [4]:

import torch
import gc
import os

from PIL import Image
from diffusers import LTXPipeline, LTXImageToVideoPipeline
from diffusers.utils import export_to_video

# Model repository
MODEL_ID = "Lightricks/LTX-Video"

# Select appropriate precision for the available GPU
DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

# Generation configuration
VIDEO_WIDTH = 512
VIDEO_HEIGHT = 320

NUM_FRAMES = 17
FPS = 24
INFERENCE_STEPS = 20

# Output directory
OUTPUT_DIR = "/content/generated_videos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Model:", MODEL_ID)
print("Precision:", DTYPE)
print("Output directory:", OUTPUT_DIR)


Disabling torchao because diffusers requires `torchao>=0.15.0` but found version 0.10.0. Upgrade it with `pip install -U 'torchao>=0.15.0'`.


Model: Lightricks/LTX-Video
Precision: torch.bfloat16
Output directory: /content/generated_videos


In [5]:
import torch
import gc
from diffusers import LTXPipeline, LTXImageToVideoPipeline

video_pipeline = None
loaded_mode = None

def load_video_model(mode):
    global video_pipeline, loaded_mode

    if video_pipeline is not None:
        del video_pipeline
        video_pipeline = None
        loaded_mode = None
        gc.collect()
        torch.cuda.empty_cache()

    pipeline_class = (
        LTXPipeline if mode == "text"
        else LTXImageToVideoPipeline
    )

    video_pipeline = pipeline_class.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        device_map="balanced",
        max_memory={
            0: "7GiB",
            "cpu": "24GiB"
        }
    )
    # Enable LTX-specific memory optimizations
    if hasattr(video_pipeline, "enable_vae_tiling"):
        video_pipeline.enable_vae_tiling()

    if hasattr(video_pipeline, "enable_vae_slicing"):
        video_pipeline.enable_vae_slicing()

    loaded_mode = mode
    return video_pipeline

In [6]:

from PIL import Image, ImageOps
import os
import torch


def generate_video(
    prompt=None,
    image=None,
    seed=42,
    num_frames=NUM_FRAMES,
    inference_steps=INFERENCE_STEPS
):

    # Determine generation mode
    if image is None:

        if not prompt or not prompt.strip():
            raise ValueError("A text prompt is required.")

        mode = "text"

    else:

        mode = "image"

        # Image-only generation uses a default animation prompt
        if not prompt or not prompt.strip():
            prompt = (
                "Animate the provided image naturally. "
                "Create smooth, realistic movement while "
                "preserving the original scene and its appearance."
            )

    print("Generation mode:", mode)
    print("Prompt:", prompt)

    # Load the appropriate pretrained pipeline
    pipe = load_video_model(mode)

    # Prepare the image if one was supplied
    if image is not None:

        if isinstance(image, str):
            image = Image.open(image)

        image = image.convert("RGB")

        # Resize and crop to the model's expected dimensions
        image = ImageOps.fit(
            image,
            (VIDEO_WIDTH, VIDEO_HEIGHT),
            method=Image.Resampling.LANCZOS
        )

    # Set random seed for reproducibility
    generator = torch.Generator(
        device="cuda"
    ).manual_seed(seed)

    print("Starting video generation...")

    # Common generation settings
    generation_args = {
        "prompt": prompt,
        "negative_prompt": (
            "blurry, distorted, jittery, low quality, "
            "inconsistent motion, flickering"
        ),
        "width": VIDEO_WIDTH,
        "height": VIDEO_HEIGHT,
        "num_frames": num_frames,
        "frame_rate": FPS,
        "num_inference_steps": inference_steps,
        "guidance_scale": 3.0,
        "generator": generator
    }

    # Generate video
    with torch.inference_mode():

        if mode == "text":

            result = pipe(**generation_args)

        else:

            result = pipe(
                image=image,
                **generation_args
            )

    # Extract generated frames
    frames = result.frames[0]

    # Create output filename
    output_path = os.path.join(
        OUTPUT_DIR,
        f"generated_video_{seed}.mp4"
    )

    # Convert frames into MP4
    export_to_video(
        frames,
        output_path,
        fps=FPS
    )

    print("Video generation complete!")
    print("Saved to:", output_path)

    return output_path


In [7]:

import google.protobuf
import sentencepiece

print("Protobuf version:", google.protobuf.__version__)
print("SentencePiece version:", sentencepiece.__version__)

# Test SentencePiece functionality
processor = sentencepiece.SentencePieceProcessor()

print("SentencePiece loaded successfully!")

Protobuf version: 7.36.2
SentencePiece version: 0.2.3
SentencePiece loaded successfully!


In [8]:
import torch
import psutil

# System RAM
ram = psutil.virtual_memory()

print(f"System RAM total: {ram.total / 1024**3:.2f} GB")
print(f"System RAM available: {ram.available / 1024**3:.2f} GB")

# GPU VRAM
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("No GPU available")

System RAM total: 12.67 GB
System RAM available: 10.66 GB
GPU: Tesla T4
GPU VRAM total: 14.56 GB


In [9]:
import torch
import psutil

ram = psutil.virtual_memory()

print("System RAM:")
print(f"Total: {ram.total / 1024**3:.2f} GB")
print(f"Available: {ram.available / 1024**3:.2f} GB")

print("\nGPU:")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(
        f"VRAM: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("No GPU detected")

System RAM:
Total: 12.67 GB
Available: 10.66 GB

GPU:
Tesla T4
VRAM: 14.56 GB


In [10]:

from google.colab import files
from IPython.display import Video, display


# Ask user for generation mode
mode = input(
    "Select generation mode:\n"
    "1. Text to Video\n"
    "2. Image to Video\n"
    "3. Image + Text to Video\n"
    "Enter 1, 2, or 3: "
).strip()

prompt = None
image = None


# Text input
if mode in ["1", "3"]:

    prompt = input(
        "Enter your video description: "
    ).strip()


# Image upload
if mode in ["2", "3"]:

    print("Upload your starting image:")

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No image was uploaded.")

    image_path = next(iter(uploaded))

    image = Image.open(image_path).convert("RGB")


# Validate input
if mode not in ["1", "2", "3"]:
    raise ValueError("Invalid generation mode.")

if mode == "1" and not prompt:
    raise ValueError("Please provide a text prompt.")

if mode == "3" and not prompt:
    raise ValueError("Please provide a text prompt.")


# Generate video
output_path = generate_video(
    prompt=prompt,
    image=image
)


# Display generated video
display(Video(output_path, embed=True))


# Download generated video
files.download(output_path)


Select generation mode:
1. Text to Video
2. Image to Video
3. Image + Text to Video
Enter 1, 2, or 3: 1
Enter your video description: a monkey eating a banana
Generation mode: text
Prompt: a monkey eating a banana


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

There are modules in LTXVideoTransformer3DModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKLLTXVideo that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

... (3 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Video generation complete!
Saved to: /content/generated_videos/generated_video_42.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>